# Node equilibrium by iteration

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

In [ ]:
from compas.datastructures import Graph
from compas.geometry import Line, Point, Vector
from compas_notebook import Viewer
from compas_notebook.config import Config

## Four anchors and one free node

In [ ]:
graph = Graph()
anchors = [graph.add_node(x=x, y=y, z=z) for x, y, z in [(0, 0, 0), (10, 0, 10), (10, 10, 0), (0, 10, 10)]]
free = graph.add_node(x=5, y=5, z=0)
for anchor in anchors:
    graph.add_edge(anchor, free)
force = 1.0

## Residual: sum of unit edge vectors times force

In [ ]:
def residual(node):
    point = Point(*graph.node_coordinates(node))
    vectors = [(Point(*graph.node_coordinates(nbr)) - point).unitized() * force for nbr in graph.neighbors(node)]
    return sum(vectors, Vector(0, 0, 0))

## Move the free node by its residual

In [ ]:
for k in range(50):
    graph.node_attributes(free, "xyz", Point(*graph.node_coordinates(free)) + residual(free))
graph.node_coordinates(free), residual(free).length

## Reactions

In [ ]:
reactions = [Line(graph.node_coordinates(node), Point(*graph.node_coordinates(node)) - residual(node)) for node in anchors]

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [5, -15, 12]
config.view.camera.target = [5, 5, 5]
viewer = Viewer(config=config)
viewer.scene.add(graph, color="#5552bb", nodesize=0.3)
for reaction in reactions:
    viewer.scene.add(reaction, color="#348f9e")
viewer.show()